# Real-ESRGAN Telegram Bot for Google Colab

โน้ตบุ๊กนี้เปลี่ยนจากเว็บ Cloudflare Tunnel เป็น **Telegram Bot** เพื่อเลี่ยงปัญหา URL เข้าไม่ได้ / DNS / tunnel หลุด

## สิ่งที่ทำได้
- ใส่ Bot Token ของคุณเองใน Colab
- ส่งรูปหรือวิดีโอเข้า Telegram Bot
- เลือกโมเดล Real-ESRGAN ผ่านปุ่มในแชท
- เลือก scale, tile, face enhance และ FPS สำหรับวิดีโอผ่านคำสั่ง/ตัวเลือก
- มี progress bar ในแชทระหว่างอัปสเกลและแทรกเฟรมวิดีโอ
- ส่ง preview + ไฟล์ผลลัพธ์กลับใน Telegram
- เลือกแทรกเฟรมวิดีโอ เช่น 24/30 FPS ไปเป็น 60 FPS ด้วย FFmpeg motion interpolation

## วิธีใช้เร็ว ๆ
1. สร้างบอทกับ `@BotFather` แล้วคัดลอก token
2. เปิด Colab เป็น GPU: `Runtime > Change runtime type > GPU`
3. รันเซลล์ติดตั้ง
4. ใส่ token ในเซลล์ config
5. รันเซลล์ Bot แล้วไปคุยกับบอทใน Telegram


In [ ]:
#@title 1) Install Real-ESRGAN, Telegram Bot SDK, and video tools
import pathlib
import subprocess
import sys

REPO_DIR = pathlib.Path('/content/Real-ESRGAN')

def run(cmd, cwd=None):
    print(f'\n$ {cmd}')
    subprocess.run(cmd, shell=True, check=True, cwd=cwd)

run('apt-get -y update >/dev/null && apt-get -y install ffmpeg wget git >/dev/null')
run(f'{sys.executable} -m pip install -U pip wheel setuptools >/dev/null')
run(f'{sys.executable} -m pip install -U "python-telegram-bot[job-queue]>=21,<22" nest_asyncio tqdm opencv-python-headless pillow numpy >/dev/null')

if not REPO_DIR.exists():
    run('git clone --depth 1 https://github.com/xinntao/Real-ESRGAN.git /content/Real-ESRGAN')

run(f'{sys.executable} -m pip install -r requirements.txt >/dev/null', cwd=str(REPO_DIR))
run(f'{sys.executable} -m pip install -e . >/dev/null', cwd=str(REPO_DIR))

# Patch basicsr for newer torchvision versions.
import site
for base in site.getsitepackages() + [site.getusersitepackages()]:
    p = pathlib.Path(base) / 'basicsr/data/degradations.py'
    if not p.exists():
        continue
    original = p.read_text()
    patched = original.replace(
        'from torchvision.transforms.functional_tensor import rgb_to_grayscale',
        'from torchvision.transforms.functional import rgb_to_grayscale',
    )
    if patched != original:
        p.write_text(patched)
        print('Patched basicsr torchvision compatibility:', p)

print('✅ Install complete. เปิด Runtime เป็น GPU จะเร็วกว่า CPU มาก')


In [ ]:
#@title 2) Config: ใส่ Telegram Bot Token ของคุณตรงนี้
# วิธีสร้าง token: เปิด Telegram > คุยกับ @BotFather > /newbot > copy token มาใส่ด้านล่าง
BOT_TOKEN = "PASTE_YOUR_TELEGRAM_BOT_TOKEN_HERE"

# ถ้าใส่ user id เฉพาะคุณ บอทจะรับเฉพาะคนใน list นี้ เช่น [123456789]
# ถ้าปล่อยว่าง [] บอทจะรับทุกคนที่มี username/token ของบอท
ALLOWED_USER_IDS = []

# จำกัดขนาดเพื่อกัน Colab/Telegram timeout; ปรับเพิ่มได้ตาม GPU/บัญชี Telegram ของคุณ
MAX_INPUT_MB = 190
MAX_OUTPUT_MB = 190

# ค่า default ตอนผู้ใช้ไม่ได้เลือกเพิ่ม
DEFAULT_OUTSCALE = 4
DEFAULT_TILE = 0          # 0 = auto / best quality, 256/512 = ประหยัด VRAM
DEFAULT_FACE_ENHANCE = False
DEFAULT_TARGET_FPS = 0      # 0 = คง FPS เดิม, 30/60/120 = แทรกเฟรมวิดีโอไปยัง FPS ที่เลือก

print('✅ Config loaded. ถ้ายังไม่ได้ใส่ BOT_TOKEN ให้แก้เซลล์นี้ก่อนรันเซลล์ Bot')


In [ ]:
#@title 3) Real-ESRGAN processing helpers for images/videos
import json
import math
import pathlib
import re
import shutil
import subprocess
import sys
import uuid

REPO_DIR = pathlib.Path('/content/Real-ESRGAN')
WORK_DIR = pathlib.Path('/content/realesrgan_telegram_jobs')
WORK_DIR.mkdir(parents=True, exist_ok=True)

MODEL_CHOICES = {
    'normal_x4': {
        'title': 'Normal x4 • RealESRGAN_x4plus',
        'model': 'RealESRGAN_x4plus',
        'scale': 4,
        'anime': False,
        'description': 'ภาพถ่ายทั่วไป คมชัด รายละเอียดสูง',
    },
    'normal_x2': {
        'title': 'Normal x2 • RealESRGAN_x2plus',
        'model': 'RealESRGAN_x2plus',
        'scale': 2,
        'anime': False,
        'description': 'เร็วกว่า เหมาะกับไฟล์ใหญ่หรือ GPU RAM น้อย',
    },
    'normal_soft_x4': {
        'title': 'Normal soft x4 • RealESRNet_x4plus',
        'model': 'RealESRNet_x4plus',
        'scale': 4,
        'anime': False,
        'description': 'ภาพถ่ายโทนนุ่ม ลดความ sharpen เกิน',
    },
    'anime_x4': {
        'title': 'Anime x4 • RealESRGAN_x4plus_anime_6B',
        'model': 'RealESRGAN_x4plus_anime_6B',
        'scale': 4,
        'anime': True,
        'description': 'อนิเมะ/ภาพวาด/มังงะ/illustration',
    },
    'anime_video_x4': {
        'title': 'Anime video x4 • realesr-animevideov3',
        'model': 'realesr-animevideov3',
        'scale': 4,
        'anime': True,
        'description': 'วิดีโออนิเมะ เน้นเสถียรภาพรายเฟรม',
    },
}

IMAGE_EXTS = {'.png', '.jpg', '.jpeg', '.webp', '.bmp'}
VIDEO_EXTS = {'.mp4', '.mov', '.mkv', '.webm', '.avi'}

FRAME_RATE_CHOICES = {
    0: 'คง FPS เดิม',
    30: 'แทรกเฟรมเป็น 30 FPS',
    60: 'แทรกเฟรมเป็น 60 FPS',
    120: 'แทรกเฟรมเป็น 120 FPS',
}


def safe_name(name: str) -> str:
    stem = pathlib.Path(name).stem or 'upload'
    stem = re.sub(r'[^A-Za-z0-9._-]+', '_', stem).strip('._') or 'upload'
    return stem[:70]


def human_mb(path: pathlib.Path) -> float:
    return path.stat().st_size / 1024 / 1024


def run_cmd(cmd, cwd=REPO_DIR):
    print(' '.join(map(str, cmd)))
    completed = subprocess.run(cmd, cwd=str(cwd), text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    if completed.returncode != 0:
        raise RuntimeError(completed.stdout[-5000:])
    return completed.stdout


def probe_video(path: pathlib.Path):
    cmd = ['ffprobe', '-v', 'error', '-select_streams', 'v:0', '-show_entries', 'stream=r_frame_rate,duration', '-of', 'json', str(path)]
    data = json.loads(subprocess.check_output(cmd, text=True))
    stream = data['streams'][0]
    fps_text = stream.get('r_frame_rate', '30/1')
    num, den = [float(x) for x in fps_text.split('/')]
    fps = num / den if den else 30.0
    duration = float(stream.get('duration') or 0.0)
    return fps, duration


def upscale_image(input_path: pathlib.Path, model_key: str, outscale: int, face_enhance: bool, tile: int, progress):
    cfg = MODEL_CHOICES[model_key]
    job = WORK_DIR / f'image_{uuid.uuid4().hex[:8]}'
    out_dir = job / 'out'
    out_dir.mkdir(parents=True, exist_ok=True)

    progress(0.08, 'เตรียมรูปภาพ')
    cmd = [sys.executable, 'inference_realesrgan.py', '-n', cfg['model'], '-i', str(input_path), '-o', str(out_dir), '--outscale', str(outscale), '--tile', str(tile)]
    if face_enhance and not cfg['anime']:
        cmd.append('--face_enhance')

    progress(0.18, f'กำลังอัปสเกลด้วย {cfg["title"]}')
    run_cmd(cmd)

    progress(0.92, 'เตรียมไฟล์ส่งกลับ')
    outputs = sorted(out_dir.glob('*'))
    if not outputs:
        raise RuntimeError('Real-ESRGAN did not create an output image.')
    final = job / f'{safe_name(input_path.name)}_{cfg["model"]}_x{outscale}.png'
    shutil.move(str(outputs[0]), final)
    progress(1.0, 'เสร็จแล้ว')
    return final, 'image'


def interpolate_video_fps(input_path: pathlib.Path, target_fps: int, source_fps: float, job: pathlib.Path, progress):
    if not target_fps or target_fps <= source_fps + 0.5:
        progress(0.98, f'คง FPS เดิม ({source_fps:.2f} FPS)')
        return input_path, source_fps

    progress(0.86, f'แทรกเฟรม {source_fps:.2f} FPS → {target_fps} FPS')
    interpolated = job / f'interpolated_{target_fps}fps.mp4'
    vf = f'minterpolate=fps={target_fps}:mi_mode=mci:mc_mode=aobmc:me_mode=bidir:vsbmc=1'
    run_cmd([
        'ffmpeg', '-y', '-i', str(input_path),
        '-vf', vf,
        '-c:v', 'libx264', '-pix_fmt', 'yuv420p', '-crf', '18', '-preset', 'medium',
        '-c:a', 'aac', '-b:a', '192k', '-movflags', '+faststart',
        str(interpolated),
    ], cwd=pathlib.Path('/content'))
    progress(0.98, f'แทรกเฟรมเสร็จแล้ว ({target_fps} FPS)')
    return interpolated, float(target_fps)


def upscale_video(input_path: pathlib.Path, model_key: str, outscale: int, face_enhance: bool, tile: int, target_fps: int, progress):
    cfg = MODEL_CHOICES[model_key]
    job = WORK_DIR / f'video_{uuid.uuid4().hex[:8]}'
    frames = job / 'frames'
    upscaled = job / 'upscaled'
    frames.mkdir(parents=True, exist_ok=True)
    upscaled.mkdir(parents=True, exist_ok=True)

    fps, duration = probe_video(input_path)
    progress(0.04, 'แยกวิดีโอเป็นเฟรม')
    run_cmd(['ffmpeg', '-y', '-i', str(input_path), '-vsync', '0', str(frames / 'frame_%08d.png')], cwd=pathlib.Path('/content'))

    frame_files = sorted(frames.glob('*.png'))
    if not frame_files:
        raise RuntimeError('No video frames were extracted.')

    # Batch processing gives real progress updates in Telegram and is safer for Colab VRAM.
    batch_size = 12 if cfg['anime'] else 8
    total_batches = math.ceil(len(frame_files) / batch_size)
    for batch_index, start in enumerate(range(0, len(frame_files), batch_size), start=1):
        batch_in = job / f'batch_in_{batch_index:04d}'
        batch_out = job / f'batch_out_{batch_index:04d}'
        batch_in.mkdir(parents=True, exist_ok=True)
        batch_out.mkdir(parents=True, exist_ok=True)

        for frame in frame_files[start:start + batch_size]:
            shutil.copy2(frame, batch_in / frame.name)

        p = 0.12 + 0.50 * ((batch_index - 1) / max(total_batches, 1))
        progress(p, f'อัปสเกลเฟรม batch {batch_index}/{total_batches}')
        cmd = [sys.executable, 'inference_realesrgan.py', '-n', cfg['model'], '-i', str(batch_in), '-o', str(batch_out), '--outscale', str(outscale), '--tile', str(tile)]
        if face_enhance and not cfg['anime']:
            cmd.append('--face_enhance')
        run_cmd(cmd)

        for produced in batch_out.glob('*'):
            target_name = produced.name.replace('_out', '') if produced.name.endswith('_out.png') else produced.name
            if not target_name.endswith('.png'):
                target_name = pathlib.Path(target_name).with_suffix('.png').name
            shutil.move(str(produced), upscaled / target_name)
        shutil.rmtree(batch_in, ignore_errors=True)
        shutil.rmtree(batch_out, ignore_errors=True)

    progress(0.66, 'รวมเฟรมกลับเป็นวิดีโอ')
    silent_video = job / 'silent.mp4'
    frame_pattern = str(upscaled / 'frame_%08d.png')
    run_cmd(['ffmpeg', '-y', '-framerate', f'{fps:.6f}', '-i', frame_pattern, '-c:v', 'libx264', '-pix_fmt', 'yuv420p', '-crf', '18', str(silent_video)], cwd=pathlib.Path('/content'))

    progress(0.76, 'ใส่เสียงเดิมกลับเข้าไฟล์')
    muxed = job / 'upscaled_with_audio.mp4'
    mux = subprocess.run(
        ['ffmpeg', '-y', '-i', str(silent_video), '-i', str(input_path), '-map', '0:v:0', '-map', '1:a?', '-c:v', 'copy', '-c:a', 'aac', '-shortest', str(muxed)],
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    if mux.returncode != 0:
        shutil.copy2(silent_video, muxed)

    processed_video, final_fps = interpolate_video_fps(muxed, target_fps, fps, job, progress)
    fps_suffix = f'_{int(round(final_fps))}fps' if target_fps and final_fps > fps + 0.5 else ''
    final = job / f'{safe_name(input_path.name)}_{cfg["model"]}_x{outscale}{fps_suffix}.mp4'
    shutil.copy2(processed_video, final)

    progress(1.0, 'เสร็จแล้ว')
    return final, 'video'


def process_media(input_path: pathlib.Path, model_key: str, outscale: int, face_enhance: bool, tile: int, target_fps: int, progress):
    ext = input_path.suffix.lower()
    if ext in IMAGE_EXTS:
        return upscale_image(input_path, model_key, outscale, face_enhance, tile, progress)
    if ext in VIDEO_EXTS:
        return upscale_video(input_path, model_key, outscale, face_enhance, tile, target_fps, progress)
    raise RuntimeError('รองรับเฉพาะรูป PNG/JPG/WEBP/BMP และวิดีโอ MP4/MOV/MKV/WEBM/AVI')

print('✅ Real-ESRGAN helpers loaded')


In [ ]:
#@title 4) Run Telegram Bot
import asyncio
import html
import pathlib
import time
import uuid

import nest_asyncio
from telegram import InlineKeyboardButton, InlineKeyboardMarkup, Update
from telegram.constants import ChatAction
from telegram.ext import Application, CallbackQueryHandler, CommandHandler, ContextTypes, MessageHandler, filters

nest_asyncio.apply()

if not BOT_TOKEN or BOT_TOKEN == 'PASTE_YOUR_TELEGRAM_BOT_TOKEN_HERE':
    raise ValueError('กรุณาใส่ BOT_TOKEN ในเซลล์ Config ก่อน')

UPLOAD_DIR = WORK_DIR / 'uploads'
UPLOAD_DIR.mkdir(parents=True, exist_ok=True)
PENDING_JOBS = {}
USER_SETTINGS = {}


def user_allowed(user_id: int) -> bool:
    return not ALLOWED_USER_IDS or user_id in ALLOWED_USER_IDS


def settings_for(user_id: int):
    return USER_SETTINGS.setdefault(user_id, {
        'outscale': DEFAULT_OUTSCALE,
        'tile': DEFAULT_TILE,
        'face_enhance': DEFAULT_FACE_ENHANCE,
        'target_fps': DEFAULT_TARGET_FPS,
    })


def progress_bar(value: float, width: int = 18) -> str:
    value = max(0.0, min(1.0, float(value)))
    filled = int(round(value * width))
    return '█' * filled + '░' * (width - filled)


def model_keyboard(job_id: str):
    rows = []
    for key, cfg in MODEL_CHOICES.items():
        rows.append([InlineKeyboardButton(cfg['title'], callback_data=f'model|{job_id}|{key}')])
    return InlineKeyboardMarkup(rows)


def settings_keyboard():
    return InlineKeyboardMarkup([
        [InlineKeyboardButton('Scale 2x', callback_data='set|outscale|2'), InlineKeyboardButton('Scale 4x', callback_data='set|outscale|4')],
        [InlineKeyboardButton('Tile Auto', callback_data='set|tile|0'), InlineKeyboardButton('Tile 256', callback_data='set|tile|256'), InlineKeyboardButton('Tile 512', callback_data='set|tile|512')],
        [InlineKeyboardButton('Face Enhance ON', callback_data='set|face|1'), InlineKeyboardButton('Face Enhance OFF', callback_data='set|face|0')],
        [InlineKeyboardButton('FPS เดิม', callback_data='set|fps|0'), InlineKeyboardButton('30 FPS', callback_data='set|fps|30')],
        [InlineKeyboardButton('60 FPS', callback_data='set|fps|60'), InlineKeyboardButton('120 FPS', callback_data='set|fps|120')],
    ])


def make_progress_callback(loop, status_message):
    last = {'time': 0, 'text': ''}

    async def edit(text):
        try:
            await status_message.edit_text(text)
        except Exception:
            pass

    def progress(value, desc):
        pct = int(max(0, min(1, value)) * 100)
        text = f'{desc}\n{progress_bar(value)} {pct}%'
        now = time.time()
        if pct >= 100 or now - last['time'] >= 4 or text != last['text']:
            last['time'] = now
            last['text'] = text
            loop.call_soon_threadsafe(asyncio.create_task, edit(text))

    return progress


async def guard(update: Update) -> bool:
    user = update.effective_user
    if user and user_allowed(user.id):
        return True
    target = update.effective_message
    if target:
        await target.reply_text('⛔ บอทนี้จำกัดผู้ใช้ไว้ใน ALLOWED_USER_IDS')
    return False


async def start(update: Update, context: ContextTypes.DEFAULT_TYPE):
    if not await guard(update):
        return
    text = (
        '✨ <b>Real-ESRGAN Upscaler Bot</b>\n\n'
        'ส่ง <b>รูป</b> หรือ <b>วิดีโอ</b> มาได้เลย แล้วบอทจะให้เลือกโมเดลอัปสเกล\n\n'
        'คำสั่ง:\n'
        '/models - ดูโมเดลทั้งหมด\n'
        '/settings - ปรับ scale/tile/face enhance/FPS วิดีโอ\n'
        '/status - ดูค่าปัจจุบัน\n'
        '/cancel - ล้างงานที่รอเลือกโมเดล\n\n'
        'Tip: วิดีโอยาวมากจะใช้เวลานาน แนะนำทดสอบคลิปสั้นก่อน'
    )
    await update.message.reply_html(text)


async def models(update: Update, context: ContextTypes.DEFAULT_TYPE):
    if not await guard(update):
        return
    lines = ['📦 <b>โมเดลที่มี</b>']
    for cfg in MODEL_CHOICES.values():
        lines.append(f'• <b>{html.escape(cfg["title"])}</b> — {html.escape(cfg["description"])}')
    await update.message.reply_html('\n'.join(lines))


async def settings(update: Update, context: ContextTypes.DEFAULT_TYPE):
    if not await guard(update):
        return
    s = settings_for(update.effective_user.id)
    text = (
        '⚙️ <b>Settings</b>\n'
        f'Output scale: <b>{s["outscale"]}x</b>\n'
        f'Tile: <b>{s["tile"]}</b>\n'
        f'Face enhance: <b>{"ON" if s["face_enhance"] else "OFF"}</b>\n'
        f'Video FPS: <b>{FRAME_RATE_CHOICES.get(s["target_fps"], str(s["target_fps"]) + " FPS")}</b>\n\n'
        'เลือกปรับได้จากปุ่มด้านล่าง'
    )
    await update.message.reply_html(text, reply_markup=settings_keyboard())


async def status(update: Update, context: ContextTypes.DEFAULT_TYPE):
    if not await guard(update):
        return
    s = settings_for(update.effective_user.id)
    await update.message.reply_text(f'ค่าใช้งานตอนนี้: scale={s["outscale"]}x, tile={s["tile"]}, face_enhance={s["face_enhance"]}, video_fps={FRAME_RATE_CHOICES.get(s["target_fps"], s["target_fps"])}')


async def cancel(update: Update, context: ContextTypes.DEFAULT_TYPE):
    if not await guard(update):
        return
    uid = update.effective_user.id
    for job_id in [j for j, data in PENDING_JOBS.items() if data['user_id'] == uid]:
        PENDING_JOBS.pop(job_id, None)
    await update.message.reply_text('🧹 ล้างงานที่รอเลือกโมเดลแล้ว')


async def save_incoming_file(update: Update, context: ContextTypes.DEFAULT_TYPE):
    if not await guard(update):
        return

    message = update.message
    user_id = update.effective_user.id
    await message.chat.send_action(ChatAction.UPLOAD_DOCUMENT)

    if message.photo:
        photo = message.photo[-1]
        if photo.file_size and photo.file_size > MAX_INPUT_MB * 1024 * 1024:
            await message.reply_text(f'ไฟล์ใหญ่เกิน {MAX_INPUT_MB} MB')
            return
        telegram_file = await photo.get_file()
        original_name = f'telegram_photo_{uuid.uuid4().hex[:8]}.jpg'
    elif message.document:
        doc = message.document
        if doc.file_size and doc.file_size > MAX_INPUT_MB * 1024 * 1024:
            await message.reply_text(f'ไฟล์ใหญ่เกิน {MAX_INPUT_MB} MB')
            return
        telegram_file = await doc.get_file()
        original_name = doc.file_name or f'telegram_document_{uuid.uuid4().hex[:8]}'
    elif message.video:
        video = message.video
        if video.file_size and video.file_size > MAX_INPUT_MB * 1024 * 1024:
            await message.reply_text(f'ไฟล์ใหญ่เกิน {MAX_INPUT_MB} MB')
            return
        telegram_file = await video.get_file()
        original_name = video.file_name or f'telegram_video_{uuid.uuid4().hex[:8]}.mp4'
    else:
        await message.reply_text('กรุณาส่งรูป วิดีโอ หรือ document ที่เป็นไฟล์รูป/วิดีโอ')
        return

    ext = pathlib.Path(original_name).suffix.lower()
    if ext not in IMAGE_EXTS and ext not in VIDEO_EXTS:
        await message.reply_text('รองรับเฉพาะ PNG/JPG/WEBP/BMP และ MP4/MOV/MKV/WEBM/AVI')
        return

    job_id = uuid.uuid4().hex[:10]
    local_path = UPLOAD_DIR / f'{job_id}_{safe_name(original_name)}{ext}'
    notice = await message.reply_text('⬇️ กำลังดาวน์โหลดไฟล์จาก Telegram...')
    await telegram_file.download_to_drive(custom_path=str(local_path))

    PENDING_JOBS[job_id] = {'user_id': user_id, 'path': local_path, 'original_name': original_name, 'created_at': time.time()}
    s = settings_for(user_id)
    await notice.edit_text(
        f'✅ รับไฟล์แล้ว: {original_name}\n'
        f'ขนาด: {human_mb(local_path):.2f} MB\n'
        f'ค่า: scale={s["outscale"]}x, tile={s["tile"]}, face_enhance={s["face_enhance"]}, video_fps={FRAME_RATE_CHOICES.get(s["target_fps"], s["target_fps"])}\n\n'
        'เลือกโมเดลที่ต้องการ:',
        reply_markup=model_keyboard(job_id),
    )



def make_safe_image_preview(output_path: pathlib.Path) -> pathlib.Path:
    from PIL import Image, ImageOps

    preview_dir = WORK_DIR / 'previews'
    preview_dir.mkdir(parents=True, exist_ok=True)
    preview_path = preview_dir / f'{output_path.stem}_preview.jpg'
    with Image.open(output_path) as im:
        im = ImageOps.exif_transpose(im)
        im.thumbnail((1280, 1280))
        if im.mode not in ('RGB', 'L'):
            im = im.convert('RGB')
        im.save(preview_path, format='JPEG', quality=92, optimize=True)
    return preview_path


async def handle_button(update: Update, context: ContextTypes.DEFAULT_TYPE):
    query = update.callback_query
    await query.answer()
    if not user_allowed(query.from_user.id):
        await query.edit_message_text('⛔ บอทนี้จำกัดผู้ใช้ไว้ใน ALLOWED_USER_IDS')
        return

    parts = query.data.split('|')
    if parts[0] == 'set':
        s = settings_for(query.from_user.id)
        if parts[1] == 'outscale':
            s['outscale'] = int(parts[2])
        elif parts[1] == 'tile':
            s['tile'] = int(parts[2])
        elif parts[1] == 'face':
            s['face_enhance'] = parts[2] == '1'
        elif parts[1] == 'fps':
            s['target_fps'] = int(parts[2])
        await query.edit_message_text(f'✅ ตั้งค่าแล้ว: scale={s["outscale"]}x, tile={s["tile"]}, face_enhance={s["face_enhance"]}, video_fps={FRAME_RATE_CHOICES.get(s["target_fps"], s["target_fps"])}')
        return

    if parts[0] != 'model' or len(parts) != 3:
        await query.edit_message_text('Callback ไม่ถูกต้อง')
        return

    job_id, model_key = parts[1], parts[2]
    job = PENDING_JOBS.get(job_id)
    if not job or job['user_id'] != query.from_user.id:
        await query.edit_message_text('งานนี้หมดอายุหรือไม่ใช่ของคุณ กรุณาส่งไฟล์ใหม่')
        return

    cfg = MODEL_CHOICES[model_key]
    s = settings_for(query.from_user.id)
    await query.edit_message_text(
        f'🚀 เริ่มงาน\nไฟล์: {job["original_name"]}\nโมเดล: {cfg["title"]}\nscale={s["outscale"]}x, tile={s["tile"]}, face_enhance={s["face_enhance"]}, video_fps={FRAME_RATE_CHOICES.get(s["target_fps"], s["target_fps"])}'
    )
    status_msg = await query.message.reply_text('คิวงานเริ่มแล้ว\n' + progress_bar(0) + ' 0%')

    loop = asyncio.get_running_loop()
    progress = make_progress_callback(loop, status_msg)

    try:
        output_path, output_kind = await asyncio.to_thread(
            process_media,
            job['path'],
            model_key,
            s['outscale'],
            s['face_enhance'],
            s['tile'],
            s['target_fps'],
            progress,
        )
        size_mb = human_mb(output_path)
        await status_msg.edit_text(f'✅ เสร็จแล้ว ขนาดไฟล์ {size_mb:.2f} MB กำลังส่งกลับ...')

        caption = f'✅ {cfg["title"]}\nScale: {s["outscale"]}x\nVideo FPS: {FRAME_RATE_CHOICES.get(s["target_fps"], s["target_fps"])}'
        if size_mb > MAX_OUTPUT_MB:
            await query.message.reply_text(f'⚠️ ไฟล์ผลลัพธ์ใหญ่ {size_mb:.2f} MB อาจส่งผ่าน Telegram ไม่ได้ อยู่ที่ Colab: {output_path}')
        else:
            if output_kind == 'image':
                preview_path = make_safe_image_preview(output_path)
                await query.message.reply_photo(photo=open(preview_path, 'rb'), caption='Preview')
            elif output_kind == 'video':
                await query.message.reply_video(video=open(output_path, 'rb'), caption='Preview', supports_streaming=True)
            await query.message.reply_document(document=open(output_path, 'rb'), filename=output_path.name, caption=caption)
        await status_msg.edit_text('🎉 ส่งผลลัพธ์เรียบร้อย')
    except Exception as exc:
        await status_msg.edit_text('❌ เกิดข้อผิดพลาด:\n' + str(exc)[-3500:])
    finally:
        PENDING_JOBS.pop(job_id, None)


async def unknown(update: Update, context: ContextTypes.DEFAULT_TYPE):
    if not await guard(update):
        return
    await update.message.reply_text('ส่งรูป/วิดีโอมาได้เลย หรือพิมพ์ /help')


async def stop_existing_bot():
    old_app = globals().get('telegram_bot_app')
    if old_app:
        await old_app.updater.stop()
        await old_app.stop()
        await old_app.shutdown()

await stop_existing_bot()

telegram_bot_app = Application.builder().token(BOT_TOKEN).read_timeout(180).write_timeout(180).connect_timeout(60).pool_timeout(60).build()
telegram_bot_app.add_handler(CommandHandler(['start', 'help'], start))
telegram_bot_app.add_handler(CommandHandler('models', models))
telegram_bot_app.add_handler(CommandHandler('settings', settings))
telegram_bot_app.add_handler(CommandHandler('status', status))
telegram_bot_app.add_handler(CommandHandler('cancel', cancel))
telegram_bot_app.add_handler(CallbackQueryHandler(handle_button))
telegram_bot_app.add_handler(MessageHandler(filters.PHOTO | filters.Document.ALL | filters.VIDEO, save_incoming_file))
telegram_bot_app.add_handler(MessageHandler(filters.ALL, unknown))

await telegram_bot_app.initialize()
await telegram_bot_app.start()
await telegram_bot_app.updater.start_polling(drop_pending_updates=True)

print('✅ Telegram Bot ทำงานแล้ว! ไปเปิด Telegram แล้วส่ง /start ให้บอทของคุณ')
print('หยุดบอท: กดปุ่ม Stop/Interrupt ใน Colab cell นี้')
await asyncio.Event().wait()
